# Sequential Sentence Classification – PubMed 20k RCT Capstone

**Ramachandra Udupa | Employee ID: 290997**

So here's the deal — medical research abstracts are dense. A doctor or a researcher skimming through a randomized controlled trial abstract usually has to read the whole thing just to figure out which part is the background, which part is the actual method, and which part is the result. That's slow, and honestly kind of annoying when you've got a hundred papers to get through.

The idea behind this project is to fix that: build a model that reads each sentence in an abstract and tags it with one of five labels — Background, Objective, Method, Result, or Conclusion. Once every sentence has a label, we can automatically reformat a wall of text into something scannable, the way a lot of good journals already do it by hand.

I'm using the PubMed 20k RCT dataset[https://github.com/Franck-Dernoncourt/pubmed-rct/tree/master] for this — about 20,000 abstracts from randomized controlled trials, already split into train/dev/test and already labeled sentence by sentence, so there's no manual labeling to do myself.

The plan is to build this up properly instead of jumping straight to the fanciest option. I'm going to try five different approaches, from the simplest possible baseline all the way up to fine-tuning an actual pretrained language model, and compare how each one does:

- **Model 0** – Naive Bayes with TF-IDF (the baseline, just to have a number to beat)
- **Model 1** – A Conv1D network with its own trained word embeddings
- **Model 2** – A frozen pretrained sentence encoder (Google's Universal Sentence Encoder) with a classifier on top
- **Model 3** – A Conv1D network, but working at the character level instead of the word level
- **Model 4** – Fine-tuning DistilBERT, a proper pretrained language model

For each one, I'll run a quick naive, default-settings version first just to see where we land, and then tune it a bit and see if it actually helps.

One more thing before we start — I'm doing all of this on my work laptop, which doesn't have a huge amount of memory to spare, and training five different models back to back adds up fast. So I've written a small helper function that clears everything out of memory once I'm done with a model, right before moving on to the next one. Nothing fancy, just good housekeeping so the notebook doesn't grind to a halt halfway through.

Alright, let's look at these one by one, shall we?

In [ ]:
import os

os.environ["TF_USE_LEGACY_KERAS"] = "1"

import gc
import string

import joblib
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, callbacks
tf.config.optimizer.set_jit(False)

tf.random.set_seed(42)

MODELS_DIR = "models"
os.makedirs(MODELS_DIR, exist_ok=True)

results = {}


def make_dataset(X, y, batch_size=32, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(1000)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


def get_callbacks(patience=3):
    return [
        callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=1, min_lr=1e-5),
    ]


def evaluate_and_record(model, name, val_ds, test_ds):
    val_loss, val_accuracy = model.evaluate(val_ds, verbose=0)
    test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
    results[name] = {
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "test_loss": test_loss,
        "test_accuracy": test_accuracy,
    }
    print(f"{name} -> Val Accuracy: {val_accuracy * 100:.2f}% | Test Accuracy: {test_accuracy * 100:.2f}%")
    return val_accuracy, test_accuracy


def free_memory():
    tf.keras.backend.clear_session()
    gc.collect()

First things first — let's load up the train, dev, and test sets. The raw files are just plain text with a label and a tab-separated sentence on each line (with `###` markers separating one abstract from the next), so I'll write a tiny parser for that instead of pulling in anything heavier.

In [2]:
def parse_pubmed_data(filename):
    with open(filename, "r") as f:
        return pd.DataFrame(
            [line.rstrip("\n").split("\t", 1) for line in f if not line.startswith("###") and not line.isspace()],
            columns=["label", "text"],
        )


train_df = parse_pubmed_data("train.txt")
val_df = parse_pubmed_data("dev.txt")
test_df = parse_pubmed_data("test.txt")

Let's take a quick look at what we're actually working with before diving in.

In [3]:
train_df.head()

,label,text
0,OBJECTIVE,To investigate the efficacy of 6 weeks of dail...
1,METHODS,A total of 125 patients with primary knee OA w...
2,METHODS,Outcome measures included pain reduction and i...
3,METHODS,Pain was assessed using the visual analog pain...
4,METHODS,Secondary outcome measures included the Wester...


Okay, model 0. Before building anything fancy, I want a baseline — something dead simple that tells me what score I'd get without putting in any real effort. TF-IDF to turn the sentences into vectors, and a Naive Bayes classifier on top. Nothing clever, just a sanity check. Let's run it and see how much we get.

In [4]:
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_df["label"])
y_val = label_encoder.transform(val_df["label"])
y_test = label_encoder.transform(test_df["label"])

X_train = train_df["text"].to_numpy()
X_val = val_df["text"].to_numpy()
X_test = test_df["text"].to_numpy()

model_0 = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", MultinomialNB()),
])
model_0.fit(X_train, y_train)

results["model_0_naive"] = {
    "val_accuracy": model_0.score(X_val, y_val),
    "test_accuracy": model_0.score(X_test, y_test),
}
print(f"Model 0 (Naive) Val Accuracy: {results['model_0_naive']['val_accuracy'] * 100:.2f}%")

Model 0 (Naive) Val Accuracy: 73.17%


Okay wow, 73.17% on the very first try, with basically zero effort. Not bad at all for a baseline. Now let's see if a bit of tuning — trying out different n-gram ranges, minimum document frequencies, and smoothing values — can push that number up a bit.

In [5]:
param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 5],
    "clf__alpha": [0.1, 0.5, 1.0],
}

random_search = RandomizedSearchCV(
    estimator=model_0,
    param_distributions=param_grid,
    n_iter=5,
    cv=3,
    n_jobs=-1,
    random_state=42,
)
random_search.fit(X_train, y_train)
model_0_tuned = random_search.best_estimator_

results["model_0_tuned"] = {
    "val_accuracy": model_0_tuned.score(X_val, y_val),
    "test_accuracy": model_0_tuned.score(X_test, y_test),
}
print(f"Best Parameters: {random_search.best_params_}")
print(f"Model 0 (Tuned) Val Accuracy: {results['model_0_tuned']['val_accuracy'] * 100:.2f}%")

joblib.dump(model_0_tuned, f"{MODELS_DIR}/model_0_naive_bayes.joblib")
del model_0, model_0_tuned, random_search
gc.collect()

Best Parameters: {'tfidf__ngram_range': (1, 1), 'tfidf__min_df': 5, 'clf__alpha': 1.0}
Model 0 (Tuned) Val Accuracy: 75.67%


0

75.67% — a solid few points better than the naive version, just from tuning the same basic pipeline. Good, that gives us a real baseline to beat.

Now let's step it up. TF-IDF treats every sentence as just a bag of words with no sense of order, so next I want to try an actual neural network that can pick up on word order and context. But before any of that — neural networks don't understand raw text, everything needs to be turned into numbers first. So let's get the data prepped into batched tensors we can actually feed into a model.

In [6]:
train_dataset = make_dataset(X_train, y_train)
val_dataset = make_dataset(X_val, y_val)
test_dataset = make_dataset(X_test, y_test)

Alright, first proper neural net. I'm turning each sentence into a sequence of word indices, running that through an embedding layer the model learns from scratch, and passing it through a Conv1D layer to pick up on local word patterns before classifying. Let's just run it with fairly basic settings first and see where we land.

In [7]:
max_vocab_length = 68000
output_seq_length = 55

text_vectorizer = layers.TextVectorization(
    max_tokens=max_vocab_length,
    output_mode="int",
    output_sequence_length=output_seq_length,
)
text_vectorizer.adapt(X_train)

token_embed = layers.Embedding(
    input_dim=len(text_vectorizer.get_vocabulary()),
    output_dim=128,
    mask_zero=True,
)

inputs = layers.Input(shape=(1,), dtype=tf.string)
x = text_vectorizer(inputs)
x = token_embed(x)
x = layers.Conv1D(filters=64, kernel_size=5, padding="same", activation="relu")(x)
x = layers.GlobalAveragePooling1D()(x)
outputs = layers.Dense(5, activation="softmax")(x)

model_1 = tf.keras.Model(inputs, outputs, name="model_1_naive")
model_1.compile(loss="sparse_categorical_crossentropy", optimizer=tf.keras.optimizers.Adam(), metrics=["accuracy"])
model_1.fit(train_dataset, epochs=3, validation_data=val_dataset)

evaluate_and_record(model_1, "model_1_naive", val_dataset, test_dataset)

Epoch 1/3


5627/5627 [==============================] - 57s 9ms/step - loss: 0.6048 - accuracy: 0.7789 - val_loss: 0.5270 - val_accuracy: 0.8133
Epoch 2/3
5627/5627 [==============================] - 23s 4ms/step - loss: 0.4372 - accuracy: 0.8468 - val_loss: 0.5283 - val_accuracy: 0.8152
Epoch 3/3
5627/5627 [==============================] - 23s 4ms/step - loss: 0.3537 - accuracy: 0.8796 - val_loss: 0.5712 - val_accuracy: 0.8080
model_1_naive -> Val Accuracy: 80.80% | Test Accuracy: 79.97%


(0.807957112789154, 0.7997013330459595)

80.80% on validation, 79.97% on test — already a solid jump over Naive Bayes, and that's still the untuned version. Let's tighten up the architecture a bit — a fresh embedding, a couple of stacked Conv1D layers, some dropout — and see what happens now.

In [8]:
fresh_token_embed = layers.Embedding(
    input_dim=len(text_vectorizer.get_vocabulary()),
    output_dim=128,
    name="fresh_token_embed",
)

inputs = layers.Input(shape=(1,), dtype=tf.string)
x = text_vectorizer(inputs)
x = fresh_token_embed(x)
x = layers.Conv1D(filters=128, kernel_size=5, padding="same", activation="relu")(x)
x = layers.Conv1D(filters=64, kernel_size=3, padding="same", activation="relu")(x)
x = layers.GlobalMaxPooling1D()(x)
x = layers.Dropout(0.2)(x)
x = layers.Dense(64, activation="relu")(x)
outputs = layers.Dense(5, activation="softmax")(x)

model_1_tuned = tf.keras.Model(inputs, outputs, name="model_1_tuned")
model_1_tuned.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    metrics=["accuracy"],
)
model_1_tuned.fit(
    train_dataset,
    epochs=10,
    validation_data=val_dataset,
    callbacks=get_callbacks(),
)

evaluate_and_record(model_1_tuned, "model_1_tuned", val_dataset, test_dataset)

model_1_tuned.save(f"{MODELS_DIR}/model_1_tuned")
del model_1, model_1_tuned, text_vectorizer, token_embed, fresh_token_embed
free_memory()

Epoch 1/10
5627/5627 [==============================] - 54s 9ms/step - loss: 0.5755 - accuracy: 0.7870 - val_loss: 0.4823 - val_accuracy: 0.8219 - lr: 0.0010
Epoch 2/10
5627/5627 [==============================] - 27s 5ms/step - loss: 0.4090 - accuracy: 0.8509 - val_loss: 0.4966 - val_accuracy: 0.8245 - lr: 0.0010
Epoch 3/10
5627/5627 [==============================] - 26s 5ms/step - loss: 0.2796 - accuracy: 0.8979 - val_loss: 0.5243 - val_accuracy: 0.8292 - lr: 2.0000e-04
Epoch 4/10
5627/5627 [==============================] - 27s 5ms/step - loss: 0.2250 - accuracy: 0.9179 - val_loss: 0.5417 - val_accuracy: 0.8296 - lr: 4.0000e-05
model_1_tuned -> Val Accuracy: 82.19% | Test Accuracy: 81.48%
INFO:tensorflow:Assets written to: models/model_1_tuned/assets


INFO:tensorflow:Assets written to: models/model_1_tuned/assets


82.19% validation, 81.48% test. Nice improvement over the naive version — the deeper architecture is clearly picking up on more.

Now for something different. Instead of training our own embeddings from scratch, what if we just borrow one that's already been trained on a massive amount of text? Google's Universal Sentence Encoder turns a whole sentence into a single 512-number vector, and it already knows a lot about language before we even start. I'll freeze it completely — no training on our data — and just stick a small classifier on top. Let's see how far that gets us with basically zero training of our own.

In [9]:
import tensorflow_hub as hub

with tf.device("/CPU:0"):
    tf_hub_embedding_layer = hub.KerasLayer(
        "https://tfhub.dev/google/universal-sentence-encoder/4",
        trainable=False,
        name="universal_sentence_encoder",
    )

    inputs = layers.Input(shape=[], dtype=tf.string)
    pretrained_embedding = tf_hub_embedding_layer(inputs)
    outputs = layers.Dense(5, activation="softmax")(pretrained_embedding)

    model_2 = tf.keras.Model(inputs, outputs, name="model_2_naive")
    model_2.compile(loss="sparse_categorical_crossentropy", optimizer=tf.keras.optimizers.Adam(), metrics=["accuracy"])
    model_2.fit(train_dataset, epochs=3, validation_data=val_dataset)

evaluate_and_record(model_2, "model_2_naive", val_dataset, test_dataset)

Epoch 1/3


   1/5627 [..............................] - ETA: 2:56:34 - loss: 1.6106 - accuracy: 0.1562

5627/5627 [==============================] - 32s 5ms/step - loss: 0.8762 - accuracy: 0.6753 - val_loss: 0.7668 - val_accuracy: 0.7124
Epoch 2/3
5627/5627 [==============================] - 29s 5ms/step - loss: 0.7589 - accuracy: 0.7122 - val_loss: 0.7437 - val_accuracy: 0.7186
Epoch 3/3
5627/5627 [==============================] - 29s 5ms/step - loss: 0.7447 - accuracy: 0.7174 - val_loss: 0.7350 - val_accuracy: 0.7212
model_2_naive -> Val Accuracy: 72.11% | Test Accuracy: 71.37%


(0.7210711240768433, 0.7136552333831787)

72.11% validation, 71.37% test. Huh — actually a bit lower than our own homegrown Conv1D from before. Interesting. A generic, general-purpose sentence embedding might be losing some of the domain-specific detail that matters here, since it was never trained on medical text specifically.

Let's give it a proper classifier head this time instead of a single Dense layer — more layers, some dropout, and enough epochs with early stopping to actually let it converge — and see if it can catch up.

In [10]:
with tf.device("/CPU:0"):
    inputs = layers.Input(shape=[], dtype=tf.string)
    pretrained_embedding = tf_hub_embedding_layer(inputs)
    x = layers.Dense(128, activation="relu")(pretrained_embedding)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation="relu")(x)
    outputs = layers.Dense(5, activation="softmax")(x)

    model_2_tuned = tf.keras.Model(inputs, outputs, name="model_2_tuned")
    model_2_tuned.compile(
        loss="sparse_categorical_crossentropy",
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        metrics=["accuracy"],
    )
    model_2_tuned.fit(
        train_dataset,
        epochs=15,
        validation_data=val_dataset,
        callbacks=get_callbacks(),
    )

evaluate_and_record(model_2_tuned, "model_2_tuned", val_dataset, test_dataset)

model_2_tuned.save(f"{MODELS_DIR}/model_2_tuned")
del model_2, model_2_tuned, tf_hub_embedding_layer, train_dataset, val_dataset, test_dataset
free_memory()

Epoch 1/15
5627/5627 [==============================] - 35s 6ms/step - loss: 0.7089 - accuracy: 0.7280 - val_loss: 0.6186 - val_accuracy: 0.7649 - lr: 0.0010
Epoch 2/15
5627/5627 [==============================] - 34s 6ms/step - loss: 0.6327 - accuracy: 0.7584 - val_loss: 0.5963 - val_accuracy: 0.7739 - lr: 0.0010
Epoch 3/15
5627/5627 [==============================] - 33s 6ms/step - loss: 0.6087 - accuracy: 0.7696 - val_loss: 0.5829 - val_accuracy: 0.7790 - lr: 0.0010
Epoch 4/15
5627/5627 [==============================] - 32s 6ms/step - loss: 0.5934 - accuracy: 0.7739 - val_loss: 0.5808 - val_accuracy: 0.7801 - lr: 0.0010
Epoch 5/15
5627/5627 [==============================] - 33s 6ms/step - loss: 0.5818 - accuracy: 0.7789 - val_loss: 0.5763 - val_accuracy: 0.7811 - lr: 0.0010
Epoch 6/15
5627/5627 [==============================] - 33s 6ms/step - loss: 0.5741 - accuracy: 0.7827 - val_loss: 0.5712 - val_accuracy: 0.7845 - lr: 0.0010
Epoch 7/15
5627/5627 [==============================

INFO:tensorflow:Assets written to: models/model_2_tuned/assets


78.97% validation, 78.30% test. Better, and tuning clearly helped, but it's still not quite catching our own Conv1D model. Fair enough — a general-purpose encoder was never going to beat something trained specifically on this data.

New idea for the next one: instead of learning at the word level, what if the model learns at the character level instead? That way it doesn't matter if it's never seen a specific medical term before — it can still pick up on patterns like suffixes, capitalization, and word shape. Let's get the data split into characters and prepped for that.

In [11]:
def split_chars(text):
    return " ".join(list(text))


train_chars = [split_chars(sentence) for sentence in X_train]
val_chars = [split_chars(sentence) for sentence in X_val]
test_chars = [split_chars(sentence) for sentence in X_test]

output_seq_char_len = 290
NUM_CHAR_TOKENS = len(string.ascii_lowercase + string.digits + string.punctuation) + 2

char_vectorizer = layers.TextVectorization(
    max_tokens=NUM_CHAR_TOKENS,
    output_sequence_length=output_seq_char_len,
    standardize="lower_and_strip_punctuation",
    name="char_vectorizer",
)
char_vectorizer.adapt(train_chars)

train_char_dataset = make_dataset(train_chars, y_train)
val_char_dataset = make_dataset(val_chars, y_val)
test_char_dataset = make_dataset(test_chars, y_test)

Let's run the character-level version with a basic setup first and see what we get.

In [12]:
char_embed = layers.Embedding(
    input_dim=NUM_CHAR_TOKENS,
    output_dim=25,
    mask_zero=False,
    name="char_embed_naive",
)

inputs = layers.Input(shape=(1,), dtype=tf.string)
x = char_vectorizer(inputs)
x = char_embed(x)
x = layers.Conv1D(filters=64, kernel_size=5, padding="same", activation="relu")(x)
x = layers.GlobalAveragePooling1D()(x)
outputs = layers.Dense(5, activation="softmax")(x)

model_3 = tf.keras.Model(inputs, outputs, name="model_3_naive")
model_3.compile(loss="sparse_categorical_crossentropy", optimizer=tf.keras.optimizers.Adam(), metrics=["accuracy"])
model_3.fit(train_char_dataset, epochs=3, validation_data=val_char_dataset)

evaluate_and_record(model_3, "model_3_naive", val_char_dataset, test_char_dataset)

Epoch 1/3
5627/5627 [==============================] - 24s 3ms/step - loss: 1.2136 - accuracy: 0.5044 - val_loss: 1.1539 - val_accuracy: 0.5354
Epoch 2/3
5627/5627 [==============================] - 16s 3ms/step - loss: 1.1264 - accuracy: 0.5441 - val_loss: 1.0782 - val_accuracy: 0.5632
Epoch 3/3
5627/5627 [==============================] - 17s 3ms/step - loss: 1.0553 - accuracy: 0.5724 - val_loss: 1.0187 - val_accuracy: 0.5892
model_3_naive -> Val Accuracy: 58.92% | Test Accuracy: 58.64%


(0.5892029404640198, 0.5863613486289978)

58.92% validation, 58.64% test. Yeah, that's rough — clearly, learning from individual characters alone, with no word-level context, is a lot harder for the model to pick up quickly. Let's tune the architecture properly this time — bigger filters, an extra conv layer, max pooling — and see if it can actually recover.

In [13]:
char_embed_tuned = layers.Embedding(
    input_dim=NUM_CHAR_TOKENS,
    output_dim=32,
    mask_zero=False,
    name="char_embed_tuned",
)

inputs = layers.Input(shape=(1,), dtype=tf.string)
x = char_vectorizer(inputs)
x = char_embed_tuned(x)
x = layers.Conv1D(filters=128, kernel_size=7, padding="same", activation="relu")(x)
x = layers.MaxPooling1D(pool_size=2)(x)
x = layers.Conv1D(filters=64, kernel_size=5, padding="same", activation="relu")(x)
x = layers.GlobalMaxPooling1D()(x)
x = layers.Dropout(0.2)(x)
x = layers.Dense(64, activation="relu")(x)
outputs = layers.Dense(5, activation="softmax")(x)

model_3_tuned = tf.keras.Model(inputs, outputs, name="model_3_tuned")
model_3_tuned.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    metrics=["accuracy"],
)
model_3_tuned.fit(
    train_char_dataset,
    epochs=10,
    validation_data=val_char_dataset,
    callbacks=get_callbacks(),
)

evaluate_and_record(model_3_tuned, "model_3_tuned", val_char_dataset, test_char_dataset)

model_3_tuned.save(f"{MODELS_DIR}/model_3_tuned")
del model_3, model_3_tuned, char_vectorizer, char_embed, char_embed_tuned
del train_char_dataset, val_char_dataset, test_char_dataset
free_memory()

Epoch 1/10


5627/5627 [==============================] - 28s 5ms/step - loss: 0.7896 - accuracy: 0.6901 - val_loss: 0.6182 - val_accuracy: 0.7655 - lr: 0.0010
Epoch 2/10
5627/5627 [==============================] - 24s 4ms/step - loss: 0.6274 - accuracy: 0.7611 - val_loss: 0.5628 - val_accuracy: 0.7882 - lr: 0.0010
Epoch 3/10
5627/5627 [==============================] - 24s 4ms/step - loss: 0.5815 - accuracy: 0.7803 - val_loss: 0.5409 - val_accuracy: 0.7964 - lr: 0.0010
Epoch 4/10
5627/5627 [==============================] - 24s 4ms/step - loss: 0.5543 - accuracy: 0.7920 - val_loss: 0.5393 - val_accuracy: 0.7965 - lr: 0.0010
Epoch 5/10
5627/5627 [==============================] - 24s 4ms/step - loss: 0.5373 - accuracy: 0.7985 - val_loss: 0.5335 - val_accuracy: 0.8011 - lr: 0.0010
Epoch 6/10
5627/5627 [==============================] - 25s 4ms/step - loss: 0.5233 - accuracy: 0.8033 - val_loss: 0.5252 - val_accuracy: 0.8049 - lr: 0.0010
Epoch 7/10
5627/5627 [==============================] - 25s 4ms

INFO:tensorflow:Assets written to: models/model_3_tuned/assets


81.58% validation, 80.79% test — that is a massive jump from 58.92%. Turns out character-level really did have something useful to say, it just needed a properly tuned network to get it out. That's now right up there with our token-level model.

Okay, last one, and this is the big gun: an actual pretrained language model. Instead of learning language from scratch on our small dataset, I'm going to start from DistilBERT — a model that's already read a huge amount of text — and adapt it to our task. First step is just getting the text tokenized the way DistilBERT expects it.

In [14]:
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification, create_optimizer

MODEL_CHECKPOINT = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

train_encodings = tokenizer(list(X_train), truncation=True, padding=True, max_length=64, return_tensors="tf")
val_encodings = tokenizer(list(X_val), truncation=True, padding=True, max_length=64, return_tensors="tf")
test_encodings = tokenizer(list(X_test), truncation=True, padding=True, max_length=64, return_tensors="tf")

train_llm_dataset = (
    tf.data.Dataset.from_tensor_slices((dict(train_encodings), y_train))
    .shuffle(1000)
    .batch(32)
    .prefetch(tf.data.AUTOTUNE)
)
val_llm_dataset = tf.data.Dataset.from_tensor_slices((dict(val_encodings), y_val)).batch(32).prefetch(tf.data.AUTOTUNE)
test_llm_dataset = tf.data.Dataset.from_tensor_slices((dict(test_encodings), y_test)).batch(32).prefetch(tf.data.AUTOTUNE)

For the first attempt, I'll freeze DistilBERT's own weights completely and only train a small classification layer on top of it, just to see what the pretrained knowledge alone gets us without touching any of its internals.

In [15]:
model_4 = TFAutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT, num_labels=5)
model_4.layers[0].trainable = False

model_4.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
model_4.fit(train_llm_dataset, epochs=2, validation_data=val_llm_dataset)

evaluate_and_record(model_4, "model_4_naive", val_llm_dataset, test_llm_dataset)

del model_4
free_memory()

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForSequenceClassification: ['vocab_transform.bias', 'vocab_layer_norm.weight', 'vocab_projector.bias', 'vocab_layer_norm.bias', 'vocab_transform.weight']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFDistilBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['pre_classifier.weight', 'pre_classifier.bias', 'classifier.weight', 'classifier.bias']
You should 

Epoch 1/2
5627/5627 [==============================] - 435s 76ms/step - loss: 0.5639 - accuracy: 0.7894 - val_loss: 0.4810 - val_accuracy: 0.8237
Epoch 2/2
5627/5627 [==============================] - 428s 76ms/step - loss: 0.5113 - accuracy: 0.8109 - val_loss: 0.4418 - val_accuracy: 0.8368
model_4_naive -> Val Accuracy: 83.68% | Test Accuracy: 82.95%


83.68% validation, 82.95% test — our best result yet, and we've barely trained anything, just a tiny layer on top of a frozen model. That says a lot about how much DistilBERT already knows.

Now let's actually unfreeze it and let the whole model fine-tune on our data properly, and see what it can really do.

In [16]:
model_4_tuned = TFAutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT, num_labels=5)

num_epochs = 3
total_train_steps = len(train_llm_dataset) * num_epochs
optimizer, lr_schedule = create_optimizer(
    init_lr=2e-5,
    num_train_steps=total_train_steps,
    num_warmup_steps=int(0.1 * total_train_steps),
    weight_decay_rate=0.01,
)

model_4_tuned.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
model_4_tuned.fit(
    train_llm_dataset,
    epochs=num_epochs,
    validation_data=val_llm_dataset,
    callbacks=[callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)],
)

evaluate_and_record(model_4_tuned, "model_4_tuned", val_llm_dataset, test_llm_dataset)

save_path = f"{MODELS_DIR}/model_4_tuned"
model_4_tuned.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

del model_4_tuned, tokenizer, train_llm_dataset, val_llm_dataset, test_llm_dataset
free_memory()

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForSequenceClassification: ['vocab_transform.bias', 'vocab_layer_norm.weight', 'vocab_projector.bias', 'vocab_layer_norm.bias', 'vocab_transform.weight']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFDistilBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['pre_classifier.weight', 'pre_classifier.bias', 'classifier.weight', 'classifier.bias']
You should 

Epoch 1/3
5627/5627 [==============================] - 1212s 214ms/step - loss: 0.4772 - accuracy: 0.8241 - val_loss: 0.3601 - val_accuracy: 0.8709
Epoch 2/3
5627/5627 [==============================] - 1210s 215ms/step - loss: 0.3267 - accuracy: 0.8809 - val_loss: 0.3494 - val_accuracy: 0.8752
Epoch 3/3
5627/5627 [==============================] - 1202s 214ms/step - loss: 0.2727 - accuracy: 0.9006 - val_loss: 0.3511 - val_accuracy: 0.8752
model_4_tuned -> Val Accuracy: 87.52% | Test Accuracy: 86.48%


87.52% validation, 86.48% test. That's the best number across everything we've tried, and it makes sense — a fully fine-tuned language model gets to adapt its entire understanding of language specifically to this task, not just the last layer.

Alright, let's line up every model we tried, naive and tuned, side by side and see the full picture.

In [17]:
results_df = pd.DataFrame(results).T.sort_values("val_accuracy", ascending=False)
results_df

,val_accuracy,test_accuracy,val_loss,test_loss
model_4_tuned,0.875248,0.864808,0.349388,0.376925
model_4_naive,0.836820,0.829534,0.441818,0.473152
model_1_tuned,0.821859,0.814767,0.482329,0.501162
model_3_tuned,0.815802,0.807931,0.498847,0.521439
model_1_naive,0.807957,0.799701,0.571232,0.588642
model_2_tuned,0.789719,0.782977,0.556496,0.580259
model_0_tuned,0.756719,0.750589,NaN,NaN
model_0_naive,0.731729,0.726564,NaN,NaN
model_2_naive,0.721071,0.713655,0.735015,0.748680
model_3_naive,0.589203,0.586361,1.018685,1.025682


And there it is — DistilBERT, fully fine-tuned, comes out on top by a clear margin, with our own tuned Conv1D and the tuned character-level model both landing in a close second tier, well ahead of the TF-IDF baseline. Every single model also improved after tuning, which is a nice sanity check that the tuning process itself was worth doing.

But a comparison table isn't really the point of any of this. The whole idea was to take a dense wall of text and turn it into something structured and scannable. So let's stop looking at accuracy numbers for a second and actually put the best model to work the way it's meant to be used.

Let's load the fine-tuned DistilBERT back up from disk — the same one we just trained, no retraining needed — along with its tokenizer. Then I'll write two small helper functions: one that splits a raw block of text into individual sentences and classifies each one, and another that takes those labeled sentences and groups the consecutive ones under the same heading, the way an actual easy-to-read abstract would be laid out.

In [18]:
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification
from nltk.tokenize import sent_tokenize
from IPython.display import Markdown, display

label_classes = sorted(train_df["label"].unique())

demo_tokenizer = AutoTokenizer.from_pretrained(f"{MODELS_DIR}/model_4_tuned")
demo_model = TFAutoModelForSequenceClassification.from_pretrained(f"{MODELS_DIR}/model_4_tuned")


def segment_abstract(raw_text):
    sentences = sent_tokenize(raw_text)
    encodings = demo_tokenizer(sentences, truncation=True, padding=True, max_length=64, return_tensors="tf")
    logits = demo_model(encodings).logits
    predicted = tf.argmax(logits, axis=1).numpy()
    return [(sentences[i], label_classes[predicted[i]]) for i in range(len(sentences))]


def format_segmented(pairs):
    sections = []
    current_label, current_chunk = None, []
    for sentence, label in pairs:
        if label != current_label:
            if current_chunk:
                sections.append(f"**{current_label.capitalize()}:** {' '.join(current_chunk)}")
            current_label, current_chunk = label, [sentence]
        else:
            current_chunk.append(sentence)
    sections.append(f"**{current_label.capitalize()}:** {' '.join(current_chunk)}")
    return "\n\n".join(sections)

All model checkpoint layers were used when initializing TFDistilBertForSequenceClassification.

All the layers of TFDistilBertForSequenceClassification were initialized from the model checkpoint at models/model_4_tuned.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFDistilBertForSequenceClassification for predictions without further training.


Now for the real test. Here's a raw, unlabeled abstract — no section headers, nothing telling us where one part ends and the next begins, just one dense paragraph the way it'd actually show up if someone pasted it straight from a paper. Let's run it through and see if the model can figure out the structure on its own.

In [22]:
raw_abstract = (
    "Investigate the efficacy of several weeks of daily low-dose oral prednisolone in improving pain, "
    "mobility, and systemic low-grade inflammation in the short term, and whether the effect is sustained "
    "in older adults with moderate to severe knee osteoarthritis (OA). A total of a specified number of "
    "patients with primary knee OA were randomized into two groups; one group received a daily dose of "
    "prednisolone while the other received a placebo. Outcome measures included pain reduction, improvement "
    "in function scores, and systemic inflammation markers. Pain was assessed using the visual analog pain "
    "scale. Secondary outcome measures included the Western Ontario and McMaster Universities Osteoarthritis "
    "Index scores, patient global assessment of knee OA severity, and walk distance. Serum levels of various "
    "biomarkers were measured. There was a clinically relevant reduction in the intervention group compared "
    "to the placebo group for knee pain, physical function, and other measures at different time points."
)

print("BEFORE - raw, unstructured:\n")
print(raw_abstract)

pairs = segment_abstract(raw_abstract)

print("\n\nAFTER - automatically segmented:\n")
display(Markdown(format_segmented(pairs)))

BEFORE - raw, unstructured:

Investigate the efficacy of several weeks of daily low-dose oral prednisolone in improving pain, mobility, and systemic low-grade inflammation in the short term, and whether the effect is sustained in older adults with moderate to severe knee osteoarthritis (OA). A total of a specified number of patients with primary knee OA were randomized into two groups; one group received a daily dose of prednisolone while the other received a placebo. Outcome measures included pain reduction, improvement in function scores, and systemic inflammation markers. Pain was assessed using the visual analog pain scale. Secondary outcome measures included the Western Ontario and McMaster Universities Osteoarthritis Index scores, patient global assessment of knee OA severity, and walk distance. Serum levels of various biomarkers were measured. There was a clinically relevant reduction in the intervention group compared to the placebo group for knee pain, physical function, and o

**Objective:** Investigate the efficacy of several weeks of daily low-dose oral prednisolone in improving pain, mobility, and systemic low-grade inflammation in the short term, and whether the effect is sustained in older adults with moderate to severe knee osteoarthritis (OA).

**Methods:** A total of a specified number of patients with primary knee OA were randomized into two groups; one group received a daily dose of prednisolone while the other received a placebo. Outcome measures included pain reduction, improvement in function scores, and systemic inflammation markers. Pain was assessed using the visual analog pain scale. Secondary outcome measures included the Western Ontario and McMaster Universities Osteoarthritis Index scores, patient global assessment of knee OA severity, and walk distance. Serum levels of various biomarkers were measured.

**Results:** There was a clinically relevant reduction in the intervention group compared to the placebo group for knee pain, physical function, and other measures at different time points.

And there it is — one dense paragraph, automatically split into Objective, Methods, and Results, without me telling it anywhere where those boundaries were. That's the whole point of this project: turn something a person has to slog through line by line into something they can scan in five seconds.

That's the technical side wrapped up — next is turning this into something the business side can actually read.